# 🌸 VisionAid - Google Colab Hub

Welcome to the official **VisionAid Google Colab Notebook**!
You can use this notebook for two primary workflows:
1. **AI Backend Server (FastAPI + GPU)**: Run YOLO11, EasyOCR & Face Recognition on Colab's free GPU and expose a public URL for your mobile phone.
2. **Android APK Compiler**: Build your `VisionAid.apk` using Buildozer in a clean Linux environment.

---

## ⚙️ Step 0: Check GPU Status
*(Tip: Go to **Runtime -> Change runtime type** and select **T4 GPU** for 10x faster AI processing)*

In [ ]:
!nvidia-smi

## 📥 Step 1: Clone VisionAid Repository

In [ ]:
import os
if not os.path.exists('VisionAid'):
    !git clone https://github.com/Sonali73724/VisionAid.git
    %cd VisionAid
else:
    %cd VisionAid
    !git pull

!ls -la

--- 
# 🚀 WORKFLOW A: Run AI Backend Server on Colab GPU
Run this section to host your VisionAid AI backend on Colab with a free public HTTPS URL.

### Step A1: Install Server Dependencies

In [ ]:
!pip install -q fastapi uvicorn ultralytics easyocr opencv-python-headless python-multipart pyngrok

### Step A2: Start Cloudflare Tunnel (Free Public HTTPS URL - No Signup Required)
This creates an instant public tunnel so your phone anywhere in the world can connect to Colab!

In [ ]:
# Download Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import time
import re

# Start cloudflared tunnel in background
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("Waiting for Cloudflare Public Tunnel URL...")
tunnel_url = None
for _ in range(25):
    line = tunnel_process.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print("\n=======================================================")
    print(f"🎉 YOUR PUBLIC VISIONAID SERVER URL: {tunnel_url}")
    print("=======================================================")
    print("\n👉 Copy this URL and paste it in your Mobile App Settings -> Server URL!")
else:
    print("Tunnel started. Check output or use pyngrok below if needed.")

### Step A3: Start VisionAid AI Server (`server.py`)

In [ ]:
!python server.py

--- 
# 📱 WORKFLOW B: Build Android APK with Buildozer
Run this section if you want to compile your `.apk` installable package.

### Step B1: Install Buildozer & Ubuntu Android Build Tools

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev cython3
!pip install --upgrade buildozer Cython virtualenv
!buildozer --version

### Step B2: Build the APK (Takes ~10-15 minutes on first run)

In [ ]:
# Compile Android Debug APK
!buildozer -v android debug

### Step B3: Download APK to Your Device

In [ ]:
import glob
from google.colab import files

apk_files = glob.glob('bin/*.apk')
if apk_files:
    print(f"Found APK: {apk_files[0]}")
    print("Downloading APK to your computer/phone...")
    files.download(apk_files[0])
else:
    print("❌ No APK found in bin/ directory. Please check build logs above.")